In [2]:
%load_ext ht.utils.ipython

The ht.utils.ipython extension is already loaded. To reload it, use:
  %reload_ext ht.utils.ipython


In [41]:
day = "2026-02-03"

from django.db.models import Exists, OuterRef, Q
from ht.apps.reports.models import LandActivity, LandJob, MTurkLock, SummaryAPICache

la_exists = LandActivity.objects.filter(summaryapicache_id=OuterRef("pk"))

la_has_creator = LandActivity.objects.filter(
    summaryapicache_id=OuterRef("pk"),
    creator_user__isnull=False,
)

turk_lock_exists = MTurkLock.objects.filter(summaryapicache_id=OuterRef("pk"))

landjob_polygon_exists = LandJob.objects.with_deleted_unapproved().filter(
    activity_id=OuterRef("pk"),
    deleted_at__isnull=True,
    polygon__isnull=False,
)

problem_qs = (
    SummaryAPICache.objects.filter(day=day)
    .annotate(
        has_land_activity=Exists(la_exists),
        has_land_activity_creator=Exists(la_has_creator),
        has_turk_lock=Exists(turk_lock_exists),
        has_polygon=Exists(landjob_polygon_exists),
    )
    .filter(Q(has_land_activity=False) | Q(has_polygon=False))
    .filter(has_turk_lock=False, has_land_activity_creator=False)
)

In [42]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN


def _to_utc_datetime(s):
    return pd.to_datetime(s, utc=True, errors="coerce")


def _meters_projection(df, lat_col="latitude", lon_col="longitude"):
    lat = df[lat_col].to_numpy(dtype=float)
    lon = df[lon_col].to_numpy(dtype=float)
    lat0 = np.nanmedian(lat)
    meters_per_deg_lat = 110540.0
    meters_per_deg_lon = 111320.0 * np.cos(np.radians(lat0))
    x = lon * meters_per_deg_lon
    y = lat * meters_per_deg_lat
    return np.column_stack([x, y])


def _haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c


def _poly_area_m2(points_xy):
    if len(points_xy) < 3:
        return 0.0
    x = points_xy[:, 0]
    y = points_xy[:, 1]
    return float(0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def _compute_distance_km(df):
    if len(df) < 2:
        return 0.0
    lat = df["latitude"].to_numpy(dtype=float)
    lon = df["longitude"].to_numpy(dtype=float)
    lat2 = np.roll(lat, -1)
    lon## What extra info we can extract (to reduce false “work done”)

# - Right now your logic says “work done” if DBSCAN finds a cluster with:
# - enough points
# - duration
# - radius
# - hull area

# - The remaining false positives are usually **transport/road** where points happen to cluster (traffic jam, parking, depot), or **GPS jitter**.

# - To enrich the decision **without drawing polygons**, add these signals per candidate cluster:

# - **Net displacement vs path length**  
#   - Transport: `net_displacement_km / path_length_km` tends to be high (straight-ish)
#   - Field work: ratio tends to be low (zig-zag / loops)

# - **Linearity (PCA variance ratio)** on XY points  
#   - Transport: ~1D → first principal component explains ~0.95+
#   - Field work: more 2D → lower ratio

# - **High-speed ratio** (if `speed` is usable)  
#   - Transport has a higher fraction of points above, say, `8 km/h` or `12 km/h`
#   - Field work often stays low speed

# - **Engine state** (`engine_state == "ON"`) ratio  
#   - Helps filter out pure jitter when tractor is off

# - Below is an **updated full notebook function** that adds these filters and returns richer cluster metrics. It still uses your DBSCAN clustering, still **does not draw polygons**, and it should reduce false positives substantially.

# ---

# Updated full code (copy/paste)

# ```python
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN


def _to_utc_datetime(s):
    return pd.to_datetime(s, utc=True, errors="coerce")


def _meters_projection(df, lat_col="latitude", lon_col="longitude"):
    lat = df[lat_col].to_numpy(dtype=float)
    lon = df[lon_col].to_numpy(dtype=float)
    lat0 = np.nanmedian(lat)
    meters_per_deg_lat = 110540.0
    meters_per_deg_lon = 111320.0 * np.cos(np.radians(lat0))
    x = lon * meters_per_deg_lon
    y = lat * meters_per_deg_lat
    return np.column_stack([x, y])


def _haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c


def _poly_area_m2(points_xy):
    if len(points_xy) < 3:
        return 0.0
    x = points_xy[:, 0]
    y = points_xy[:, 1]
    return float(0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def _compute_path_length_km(df):
    if len(df) < 2:
        return 0.0
    lat = df["latitude"].to_numpy(dtype=float)
    lon = df["longitude"].to_numpy(dtype=float)
    lat2 = np.roll(lat, -1)
    lon2 = np.roll(lon, -1)
    dist = _haversine_km(lat[:-1], lon[:-1], lat2[:-1], lon2[:-1])
    dist = np.nan_to_num(dist, nan=0.0, posinf=0.0, neginf=0.0)
    return float(dist.sum())


def _compute_net_displacement_km(df):
    if len(df) < 2:
        return 0.0
    lat1 = float(df["latitude"].iloc[0])
    lon1 = float(df["longitude"].iloc[0])
    lat2 = float(df["latitude"].iloc[-1])
    lon2 = float(df["longitude"].iloc[-1])
    return float(_haversine_km(lat1, lon1, lat2, lon2))


def _compute_moving_ratio(df):
    if "speed" not in df.columns:
        return None
    spd = pd.to_numeric(df["speed"], errors="coerce").fillna(0)
    return float((spd > 0.5).mean())


def _compute_high_speed_ratio(df, high_kmh=12.0):
    if "speed" not in df.columns:
        return None
    spd = pd.to_numeric(df["speed"], errors="coerce").fillna(0)
    return float((spd > float(high_kmh)).mean())


def _compute_engine_on_ratio(df):
    if "engine_state" not in df.columns:
        return None
    es = df["engine_state"].astype(str).str.upper()
    return float((es == "ON").mean())


def _split_sessions(df, max_gap_min=20.0):
    if max_gap_min is None:
        return [df]
    gaps = df["location_time"].diff().dt.total_seconds().div(60.0)
    session_id = (gaps.fillna(0) > float(max_gap_min)).cumsum()
    return [g.reset_index(drop=True) for _, g in df.groupby(session_id, sort=True)]


def _safe_hull_area_ha(sub_xy):
    if sub_xy is None or len(sub_xy) < 3:
        return 0.0

    sub_xy_u = np.unique(np.round(sub_xy, 2), axis=0)
    if len(sub_xy_u) < 3:
        return 0.0

    # reject near-1D very early
    if float(np.std(sub_xy_u[:, 0])) < 0.5 or float(np.std(sub_xy_u[:, 1])) < 0.5:
        return 0.0

    try:
        from scipy.spatial import ConvexHull, QhullError

        hull = ConvexHull(sub_xy_u, qhull_options="QJ")
        hull_xy = sub_xy_u[hull.vertices]
        return _poly_area_m2(hull_xy) / 10000.0
    except (QhullError, ValueError):
        return 0.0


def _pca_linearity_ratio(xy):
    # 1.0 => nearly a line, 0.5 => more 2D spread
    if xy is None or len(xy) < 3:
        return 1.0
    x = xy - np.mean(xy, axis=0, keepdims=True)
    cov = np.cov(x.T)
    if not np.isfinite(cov).all():
        return 1.0
    vals = np.linalg.eigvalsh(cov)
    vals = np.sort(vals)[::-1]
    denom = float(vals.sum())
    if denom <= 0:
        return 1.0
    return float(vals[0] / denom)


def identify_work_done_for_asset_day_inline(
    df_raw,
    *,
    day=None,
    resample_rule="2min",
    max_session_gap_min=20.0,
    eps_m=20.0,
    min_samples=10,
    min_cluster_duration_min=15.0,
    max_cluster_radius_m=120.0,
    min_cluster_area_ha=0.2,
    min_points_total=40,
    max_linearity_ratio=0.92,          # <= this is more “2D field-like”
    max_displacement_path_ratio=0.55,  # net_disp / path_len should be small-ish for field work
    max_high_speed_ratio=0.35,         # transport tends to have high high-speed ratio
    high_speed_kmh=12.0,
    min_engine_on_ratio=0.3,           # helps reject GPS jitter when off
):
    df = df_raw.copy()
    df["location_time"] = _to_utc_datetime(df["location_time"])
    df = df.dropna(subset=["location_time", "latitude", "longitude"]).sort_values("location_time")

    if day is not None:
        day_dt = pd.to_datetime(day, utc=True, errors="coerce")
        if pd.isna(day_dt):
            return {"worked": False, "reason_codes": ["bad_day"], "clusters": []}
        start = day_dt.normalize()
        end = start + pd.Timedelta(days=1)
        df = df[(df["location_time"] >= start) & (df["location_time"] < end)]

    points_total = int(len(df))
    if points_total == 0:
        return {
            "worked": False,
            "reason_codes": ["no_points"],
            "points_total": 0,
            "points_used": 0,
            "path_km": 0.0,
            "moving_ratio": None,
            "clusters": [],
        }

    if resample_rule is not None:
        df = (
            df.set_index("location_time")
            .resample(resample_rule).last()
            .reset_index()
            .dropna(subset=["location_time", "latitude", "longitude"])
            .sort_values("location_time")
            .reset_index(drop=True)
        )

    points_used = int(len(df))
    moving_ratio = _compute_moving_ratio(df)
    path_km = _compute_path_length_km(df)

    reason_codes = []
    if points_used < int(min_points_total):
        reason_codes.append("too_few_points")
    if moving_ratio is not None and moving_ratio < 0.02:
        reason_codes.append("low_moving_ratio")

    eps_list = [float(eps_m), float(eps_m) * 1.5, float(eps_m) * 2.5]
    eps_list = [e for e in eps_list if e > 0]
    eps_list = [e if e <= 150 else 150.0 for e in eps_list]

    sessions = _split_sessions(df, max_gap_min=max_session_gap_min)

    clusters_out = []

    for sess in sessions:
        if len(sess) < max(3, int(min_samples), int(min_points_total)):
            continue

        xy = _meters_projection(sess)

        ms_list = [int(min_samples), max(5, int(min_samples) // 2), 3]
        ms_list = [ms for ms in ms_list if ms <= len(sess)]

        for ms_try in ms_list:
            found_any = False
            for eps_try in eps_list:
                labels = DBSCAN(eps=float(eps_try), min_samples=int(ms_try)).fit_predict(xy)

                for cl in sorted(set(int(x) for x in labels)):
                    if cl == -1:
                        continue

                    idx = np.where(labels == cl)[0]
                    sub = sess.iloc[idx]
                    if len(sub) < int(min_points_total):
                        continue

                    start_t = sub["location_time"].min()
                    end_t = sub["location_time"].max()
                    duration_min = float((end_t - start_t).total_seconds() / 60.0)
                    if duration_min < float(min_cluster_duration_min):
                        continue

                    centroid_lat = float(sub["latitude"].mean())
                    centroid_lon = float(sub["longitude"].mean())

                    sub_xy = _meters_projection(sub)
                    cxy = _meters_projection(
                        pd.DataFrame({"latitude": [centroid_lat], "longitude": [centroid_lon]})
                    )[0]
                    radius_m = float(np.sqrt(((sub_xy - cxy) ** 2).sum(axis=1)).max())
                    if radius_m > float(max_cluster_radius_m):
                        continue

                    hull_area_ha = _safe_hull_area_ha(sub_xy)
                    if hull_area_ha < float(min_cluster_area_ha):
                        continue

                    # Enrichment filters (transport/jitter rejection)
                    pca_ratio = _pca_linearity_ratio(sub_xy)
                    if pca_ratio > float(max_linearity_ratio):
                        continue

                    path_len = _compute_path_length_km(sub)
                    net_disp = _compute_net_displacement_km(sub)
                    disp_path_ratio = (net_disp / path_len) if path_len > 0 else 1.0
                    if disp_path_ratio > float(max_displacement_path_ratio):
                        continue

                    hs_ratio = _compute_high_speed_ratio(sub, high_kmh=high_speed_kmh)
                    if hs_ratio is not None and hs_ratio > float(max_high_speed_ratio):
                        continue

                    eng_on_ratio = _compute_engine_on_ratio(sub)
                    if eng_on_ratio is not None and eng_on_ratio < float(min_engine_on_ratio):
                        continue

                    clusters_out.append(
                        dict(
                            points=int(len(sub)),
                            start_time=start_t,
                            end_time=end_t,
                            duration_min=duration_min,
                            centroid_lat=centroid_lat,
                            centroid_lon=centroid_lon,
                            radius_m=radius_m,
                            hull_area_ha=float(hull_area_ha),
                            pca_linearity_ratio=float(pca_ratio),
                            path_km=float(path_len),
                            net_disp_km=float(net_disp),
                            disp_path_ratio=float(disp_path_ratio),
                            high_speed_ratio=hs_ratio,
                            engine_on_ratio=eng_on_ratio,
                            eps_m=float(eps_try),
                            min_samples=int(ms_try),
                        )
                    )
                    found_any = True

                if found_any:
                    break
            if found_any:
                break

    clusters_out.sort(key=lambda c: (c["duration_min"], c["points"], c["hull_area_ha"]), reverse=True)

    worked = bool(clusters_out)
    reason_codes.append("dbscan_cluster_found" if worked else "no_dbscan_cluster")

    return {
        "worked": bool(worked),
        "reason_codes": reason_codes,
        "points_total": points_total,
        "points_used": points_used,
        "path_km": float(path_km),
        "moving_ratio": moving_ratio,
        "clusters": clusters_out,
    }

In [43]:
import pandas as pd
from ht.apps.real_time_data.models import Asset
from ht.apps.timescaledb.models import LiveTrackingData

start = datetime.strptime(day, "%Y-%m-%d").replace(tzinfo=timezone.utc)
end = start + timedelta(days=1)

flagged = []
checked = 0

for sac in problem_qs.only("id", "tractor_id", "day"):
    checked += 1
    tractor_id = str(sac.tractor_id)

    asset = (
        Asset.objects
        .filter(tractor_id=tractor_id)
        .order_by("created_at")
        .last()
    )
    if not asset:
        continue

    ltd_rows = list(
        LiveTrackingData.objects.filter(
            asset_id=asset.asset_uuid,
            location_time__gte=start,
            location_time__lt=end,
        )
        .values("location_time", "latitude", "longitude", "speed", "engine_state")
        .order_by("location_time")
    )
    if not ltd_rows:
        continue

    df = pd.DataFrame(ltd_rows)

    diag = identify_work_done_for_asset_day_inline(
        df,
        day=day,
        resample_rule="2min",
        eps_m=20.0,
        min_samples=10,
        min_cluster_duration_min=15.0,
        max_cluster_radius_m=120.0,
        min_cluster_area_ha=0.2,
        min_points_total=40,
        max_linearity_ratio=0.92,
        max_displacement_path_ratio=0.55,
        max_high_speed_ratio=0.35,
        high_speed_kmh=12.0,
        min_engine_on_ratio=0.3,
    )

    if not diag["worked"]:
        continue

    top = diag["clusters"][0] if diag["clusters"] else {}

    print(f"{tractor_id} -> YES (work detected)")

    flagged.append(
        {
            "sac_id": sac.id,
            "tractor_id": tractor_id,
            "day": str(sac.day),
            "asset_uuid": str(asset.asset_uuid),
            "reason_codes": ";".join(diag["reason_codes"]),
            "points_used": diag["points_used"],
            "path_km": diag.get("path_km"),
            "moving_ratio": diag.get("moving_ratio"),
            "top_cluster_duration_min": top.get("duration_min"),
            "top_cluster_area_ha": top.get("hull_area_ha"),
            "top_cluster_radius_m": top.get("radius_m"),
            "top_cluster_linearity": top.get("pca_linearity_ratio"),
            "top_cluster_disp_path_ratio": top.get("disp_path_ratio"),
            "top_cluster_high_speed_ratio": top.get("high_speed_ratio"),
            "top_cluster_engine_on_ratio": top.get("engine_on_ratio"),
            "top_cluster_path_km": top.get("path_km"),
            "top_cluster_net_disp_km": top.get("net_disp_km"),
            "dbscan_eps_m": top.get("eps_m"),
            "dbscan_min_samples": top.get("min_samples"),
        }
    )

flagged_df = pd.DataFrame(flagged)

print("Checked:", checked)
print("Flagged (work detected):", len(flagged_df))

if len(flagged_df):
    flagged_df = flagged_df.sort_values(
        ["top_cluster_area_ha", "top_cluster_duration_min", "points_used"],
        ascending=False,
    )
flagged_df.head(30)

706965 -> YES (work detected)
507024 -> YES (work detected)
507649 -> YES (work detected)
507561 -> YES (work detected)
507395 -> YES (work detected)
706665 -> YES (work detected)
504745 -> YES (work detected)
507447 -> YES (work detected)
504968 -> YES (work detected)
507555 -> YES (work detected)
504974 -> YES (work detected)
504997 -> YES (work detected)
503174 -> YES (work detected)
506386 -> YES (work detected)
507632 -> YES (work detected)
507807 -> YES (work detected)
706390 -> YES (work detected)
507676 -> YES (work detected)
504938 -> YES (work detected)
706676 -> YES (work detected)
504958 -> YES (work detected)
504975 -> YES (work detected)
706728 -> YES (work detected)
507185 -> YES (work detected)
507025 -> YES (work detected)
507844 -> YES (work detected)
706784 -> YES (work detected)
707004 -> YES (work detected)
504914 -> YES (work detected)
507589 -> YES (work detected)
507110 -> YES (work detected)
706449 -> YES (work detected)
706430 -> YES (work detected)
507435 -> 

,sac_id,tractor_id,day,asset_uuid,reason_codes,points_used,path_km,moving_ratio,top_cluster_duration_min,top_cluster_area_ha,top_cluster_radius_m,top_cluster_linearity,top_cluster_disp_path_ratio,top_cluster_high_speed_ratio,top_cluster_engine_on_ratio,top_cluster_path_km,top_cluster_net_disp_km,dbscan_eps_m,dbscan_min_samples
22,1767536,706728,2026-02-03,c5c23e17-aca6-5aca-b571-ab768a59b8ec,dbscan_cluster_found,365,39.139707,0.558904,540.0,2.145029,117.255998,0.811403,0.000444,0.000000,1.000000,13.403520,0.005952,20.0,5
34,1768074,507136,2026-02-03,6fa79d10-e51e-11ef-9f33-8fe47a0a8e13,dbscan_cluster_found,237,18.824589,0.358650,152.0,1.799881,99.695681,0.780297,0.017262,0.000000,1.000000,5.539422,0.095620,50.0,10
25,1767863,507844,2026-02-03,7022cc80-96d3-11f0-a303-4fbcf719bfa0,dbscan_cluster_found,306,50.948830,0.758170,308.0,1.442543,114.206789,0.872174,0.013119,0.000000,1.000000,6.578875,0.086310,50.0,10
15,1767452,507807,2026-02-03,8126afc0-8566-11f0-a303-4fbcf719bfa0,dbscan_cluster_found,226,10.306224,0.367257,424.0,1.154661,110.113469,0.832909,0.029555,0.000000,0.666667,2.868592,0.084780,50.0,10
28,1770228,504914,2026-02-03,034d00a0-e788-11ed-8e76-0bde7aa14120,dbscan_cluster_found,133,23.652255,0.857143,206.0,1.097172,101.408744,0.902650,0.051485,0.000000,1.000000,2.554249,0.131507,30.0,10
31,1767878,706449,2026-02-03,032f7935-8278-5ea0-b395-8f5db657e129,dbscan_cluster_found,262,32.647472,0.290076,532.0,0.980022,116.027154,0.788477,0.452707,0.000000,0.666667,0.325544,0.147376,50.0,10
12,1772183,503174,2026-02-03,066f4f60-88e1-11ec-a410-d301109ec991,dbscan_cluster_found,190,24.061611,0.610526,130.0,0.914594,88.617944,0.820097,0.001037,0.000000,1.000000,3.682095,0.003818,30.0,10
4,1767771,507395,2026-02-03,415ba0e0-22be-11f0-9f33-8fe47a0a8e13,dbscan_cluster_found,238,6.352101,0.184874,574.0,0.820923,70.293035,0.613245,0.022147,0.000000,0.882353,1.020955,0.022611,30.0,10
2,1767578,507649,2026-02-03,be7441c0-3283-11f0-9f33-8fe47a0a8e13,dbscan_cluster_found,236,16.312288,0.343220,130.0,0.815033,97.581082,0.729096,0.033470,0.000000,1.000000,2.724876,0.091201,30.0,10
19,1768377,706676,2026-02-03,3fe6d2b4-174c-50c4-9a49-40b6981e469f,dbscan_cluster_found,229,8.018930,0.131004,198.0,0.794532,78.287907,0.744378,0.001889,0.000000,0.977273,3.444480,0.006508,30.0,10
